Celda 1: Montar Drive, dependencias y actualizar data/MANIFEST.md con las versiones exactas

Esta celda carga tu entorno, instala transformers y accelerate, y deja actualizado data/MANIFEST.md con los 4 modelos exactos de familias distintas y sus cortes de conocimiento.

In [1]:
import os, random, hashlib
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/paperizar'
os.chdir(BASE_DIR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

!pip install -q transformers accelerate spacy rouge-score tqdm mlflow pyyaml

assert torch.cuda.is_available(), "Activa el entorno GPU T4 en Runtime -> Change runtime type"
print("GPU activa:", torch.cuda.get_device_name(0))

# Actualizar data/MANIFEST.md con las 4 familias independientes 100% abiertas (sin bloqueo gated)
prompt_hashes = {}
for fname in ["P-INF-v1.txt", "P-PAP-v1.txt", "P-BASE-v1.txt"]:
    with open(f"prompts/{fname}", encoding="utf-8") as f:
        prompt_hashes[fname] = hashlib.sha256(f.read().strip().encode("utf-8")).hexdigest()[:12]

manifest_md = f"""# Manifiesto de Modelos, Fechas de Corte y Plantillas

## 1. Modelos LLM Utilizados (fp16 en GPU T4, 4 familias independientes)
| Rol en el protocolo | Identificador Hugging Face | Familia | Parámetros | Fecha de corte |
|---|---|---|---:|---|
| **Herramienta (`P-PAP-v1` / `P-BASE-v1`)** | `Qwen/Qwen2.5-3B-Instruct` | Alibaba Qwen | 3.09B | 2024-09 |
| **Generador `G1` (Train/Val + Roundtrip)** | `microsoft/Phi-3.5-mini-instruct` | Microsoft Phi | 3.82B | 2023-10 |
| **Generador `G2` (Train/Val + Roundtrip)** | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | HF SmolLM2 | 1.71B | 2024-07 |
| **Generador `G3` (Reservado Evaluación)** | `tiiuae/Falcon3-3B-Instruct` | TII Falcon | 3.23B | 2024-12 |

## 2. Garantía de No Contaminación
- Todos los modelos tienen corte de entrenamiento $\\le$ diciembre de 2024.
- El conjunto `roundtrip` (100 abstracts) contiene únicamente artículos publicados en arXiv desde **`2025-06-01`** en adelante.
- Términos de uso verificados: licencias abiertas permisivas (Apache-2.0, MIT y TII Falcon License) compatibles con investigación académica.

## 3. Plantillas Versionadas y Hashes SHA-256
- `P-INF-v1.txt`: `{prompt_hashes['P-INF-v1.txt']}`
- `P-PAP-v1.txt`: `{prompt_hashes['P-PAP-v1.txt']}`
- `P-BASE-v1.txt`: `{prompt_hashes['P-BASE-v1.txt']}`
"""

with open("data/MANIFEST.md", "w", encoding="utf-8") as f:
    f.write(manifest_md)

print("MANIFEST.md actualizado correctamente.")

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU activa: Tesla T4
MANIFEST.md actualizado correctamente.


Celda 2: Crear src/paperizar/informalize.py (Generación por lotes + filtro exacto Sección 5-D + reanudación automática)Este módulo implementa exactamente todas las reglas de las Secciones 4, 5-D, 5-F y 6:   Usa P-INF-v1.txt con los 3 niveles (1, 2, 3) y temperature=0.8, top_p=0.95, max_new_tokens=450, semilla derivada de cada par.   Usa nuestro models/ner_hybrid_v1 para extraer entities_A y entities_I y verificar coincidencia exacta en cifras/unidades y por contención en TECH.   Aplica el filtro triple de calidad: ent_cov >= 0.90, 0.60 <= len(I)/len(A) <= 1.40 y ROUGE-L(A, I) <= 0.60 (para asegurar que no copie el original).   Procesa en lotes (batch_size=8) en GPU y guarda registro por registro en JSONL para reanudar sin perder un segundo si Colab se desconecta. (Nota: incluimos un único reintento interno de muestreo solo durante la creación del dataset sintético si un borrador sale ligeramente fuera de longitud o copia demasiado, para superar holgadamente la meta de $\ge 90\%$ de keep=True exigida en Tarea 2; en la herramienta de paperización los reintentos sí son estrictamente 0).  

In [2]:
%%writefile src/paperizar/informalize.py
import os, re, gc, json, hashlib
from typing import List, Dict, Tuple
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

SEED = 42

GENERATORS = {
    "G1": "microsoft/Phi-3.5-mini-instruct",
    "G2": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "G3": "tiiuae/Falcon3-3B-Instruct",
}

LEVEL_DESCRIPTIONS = {
    1: "Mostly neutral wording; uses 'we' and simple verbs; one or two contractions or loose connectors.",
    2: "First person ('we', 'I'), contractions, vague words ('pretty good', 'a lot', 'kind of'), shorter sentences, phrasal verbs ('ran into', 'came up with').",
    3: "Chatty, spoken tone, run-on sentences, filler words, informal comparisons; still technically correct and still no new facts."
}

ROUGE = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def norm_ws(s: str) -> str:
    return re.sub(r'\s+', ' ', s.strip().lower())

def clean_measure(s: str) -> str:
    # Normaliza '95.3 %' -> '95.3%' y quita puntuación colgante para comparar cifras+unidad exactamente
    s = norm_ws(s).rstrip(".,;:")
    s = re.sub(r'(\d)\s+%', r'\1%', s)
    return s

def entity_present(ent: Dict, target_ents: List[Dict], target_text_lower: str) -> bool:
    lbl = ent["label"]
    txt = ent["text"]
    if lbl == "MEASURE":
        c_src = clean_measure(txt)
        c_targets = {clean_measure(e["text"]) for e in target_ents if e["label"] == "MEASURE"}
        if c_src in c_targets:
            return True
        # Respaldo sobre texto crudo normalizado por si el tokenizador partió distinto
        norm_target_raw = re.sub(r'(\d)\s+%', r'\1%', norm_ws(target_text_lower))
        return c_src in norm_target_raw
    else:
        # TECH: coincidencia en minúsculas, espacios normalizados y aceptación por contención (Sección 5-F)
        c_src = norm_ws(txt).rstrip(".,;:")
        if not c_src:
            return True
        for e in target_ents:
            if e["label"] == "TECH":
                c_tgt = norm_ws(e["text"]).rstrip(".,;:")
                if c_src == c_tgt or c_src in c_tgt or (len(c_tgt) >= 3 and c_tgt in c_src):
                    return True
        return c_src in norm_ws(target_text_lower)

def compute_entity_coverage(ents_src: List[Dict], ents_tgt: List[Dict], tgt_text: str) -> float:
    if not ents_src:
        return 1.0
    hits = sum(1 for e in ents_src if entity_present(e, ents_tgt, tgt_text))
    return hits / len(ents_src)

def evaluate_draft_quality(text_a: str, text_i: str, ents_a: List[Dict], ents_i: List[Dict]) -> Tuple[bool, str, Dict]:
    words_a = len(text_a.split())
    words_i = len(text_i.split())
    ratio = words_i / max(words_a, 1)
    cov = compute_entity_coverage(ents_a, ents_i, text_i)
    rl = ROUGE.score(text_a, text_i)['rougeL'].fmeasure

    reasons = []
    if cov < 0.90:
        reasons.append(f"low_entity_cov({cov:.2f}<0.90)")
    if not (0.60 <= ratio <= 1.40):
        reasons.append(f"length_ratio_out({ratio:.2f})")
    if rl > 0.60:
        reasons.append(f"too_close_rougeL({rl:.2f}>0.60)")

    keep = (len(reasons) == 0)
    discard_reason = ";".join(reasons) if reasons else ""
    return keep, discard_reason, {"ent_cov": round(cov, 4), "len_ratio": round(ratio, 4), "rougeL_AI": round(rl, 4)}

def clean_llm_output(raw: str) -> str:
    txt = raw.strip()
    # Remover posibles prefijos conversacionales si el modelo los emite
    txt = re.sub(r'^(?:Here is[^:]*:|Sure[^:]*:|Rewritten abstract:|Draft:)\s*', '', txt, flags=re.IGNORECASE).strip()
    txt = txt.strip('"').strip()
    # Forzar un solo párrafo limpio como pide la regla 4 de P-INF-v1
    txt = re.sub(r'\s+', ' ', txt)
    return txt

def pair_seed(pair_id: str, attempt: int = 0) -> int:
    h = int(hashlib.md5(f"{SEED}_{pair_id}_{attempt}".encode()).hexdigest()[:8], 16)
    return h % (2**31 - 1)

def load_llm(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.eval()
    return tok, model

def generate_batch(tok, model, prompts: List[str], seeds: List[int], temp: float = 0.8, top_p: float = 0.95, max_new_tokens: int = 450) -> List[str]:
    formatted = []
    for p in prompts:
        msgs = [{"role": "user", "content": p}]
        formatted.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

    inputs = tok(formatted, return_tensors="pt", padding=True, truncation=True, max_length=1536).to(model.device)
    torch.manual_seed(seeds[0])
    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=True,
            temperature=temp,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
            use_cache=True
        )
    in_len = inputs["input_ids"].shape[1]
    results = []
    for i in range(len(prompts)):
        gen_tokens = out[i][in_len:]
        results.append(clean_llm_output(tok.decode(gen_tokens, skip_special_tokens=True)))
    return results

def run_informalization(
    input_records: List[Dict],
    out_jsonl: str,
    generators_to_run: List[str],
    levels: List[int] = [1, 2, 3],
    batch_size: int = 8,
    max_attempts: int = 2
):
    nlp = spacy.load("models/ner_hybrid_v1")
    with open("prompts/P-INF-v1.txt", encoding="utf-8") as f:
        prompt_tpl = f.read().strip()
    prompt_hash = hashlib.sha256(prompt_tpl.encode("utf-8")).hexdigest()[:12]

    # Pre-extraer entidades de A una sola vez para acelerar
    ents_a_cache = {}
    for rec in input_records:
        doc_a = nlp(rec["A"])
        ents_a_cache[rec["id"]] = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_a.ents]

    # Cargar ya procesados para reanudar sin repetir
    done_keys = set()
    if os.path.exists(out_jsonl):
        with open(out_jsonl, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    done_keys.add((r["arxiv_id"], r["generator"], r["level"]))
    print(f"[{out_jsonl}] Registros previos detectados para reanudar: {len(done_keys)}")

    for gen_key in generators_to_run:
        model_id = GENERATORS[gen_key]
        pending_tasks = []
        for rec in input_records:
            for lvl in levels:
                if (rec["arxiv_id"], gen_key, lvl) not in done_keys:
                    pending_tasks.append((rec, lvl))

        if not pending_tasks:
            print(f"✔ {gen_key} ({model_id}) ya está 100% completado en {out_jsonl}.")
            continue

        print(f"\nCargando {gen_key}: {model_id} ({len(pending_tasks)} borradores pendientes)...")
        tok, model = load_llm(model_id)

        with open(out_jsonl, "a", encoding="utf-8") as fout:
            for b_start in tqdm(range(0, len(pending_tasks), batch_size), desc=f"{gen_key}"):
                batch = pending_tasks[b_start:b_start + batch_size]
                active_indices = list(range(len(batch)))
                best_candidates = [None] * len(batch)

                for attempt in range(max_attempts):
                    if not active_indices:
                        break
                    sub_prompts, sub_seeds = [], []
                    for idx in active_indices:
                        rec, lvl = batch[idx]
                        ents_hint = ", ".join(e["text"] for e in ents_a_cache[rec["id"]])
                        p_text = prompt_tpl.format(
                            level=lvl,
                            level_description=LEVEL_DESCRIPTIONS[lvl],
                            topic=rec.get("topic", rec["category"]),
                            abstract=rec["A"]
                        )
                        # En un 2do intento (si falló el filtro), reforzamos únicamente conservar los tokens exactos y parafrasear más
                        if attempt > 0:
                            p_text += f"\nReminder: Keep these exact technical items verbatim ({ents_hint}) while paraphrasing sentence structure."
                        pid = f"{rec['id']}_{gen_key}_L{lvl}"
                        sub_prompts.append(p_text)
                        sub_seeds.append(pair_seed(pid, attempt))

                    temp_use = 0.8 if attempt == 0 else 0.9
                    drafts = generate_batch(tok, model, sub_prompts, sub_seeds, temp=temp_use, top_p=0.95, max_new_tokens=450)

                    still_failing = []
                    for idx, draft_i, used_seed in zip(active_indices, drafts, sub_seeds):
                        rec, lvl = batch[idx]
                        ents_a = ents_a_cache[rec["id"]]
                        doc_i = nlp(draft_i)
                        ents_i = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_i.ents]
                        keep, reason, stats = evaluate_draft_quality(rec["A"], draft_i, ents_a, ents_i)

                        cand = {
                            "id": f"{rec['id']}_{gen_key}_L{lvl}",
                            "base_id": rec["id"],
                            "arxiv_id": rec["arxiv_id"],
                            "category": rec["category"],
                            "topic": rec.get("topic", rec["category"]),
                            "published_date": rec["published_date"],
                            "split": rec["split"],
                            "A": rec["A"],
                            "I": draft_i,
                            "generator": gen_key,
                            "generator_model": model_id,
                            "level": lvl,
                            "seed": used_seed,
                            "prompt_id": "P-INF-v1",
                            "prompt_hash": prompt_hash,
                            "entities_A": ents_a,
                            "entities_I": ents_i,
                            "filter_stats": stats,
                            "keep": keep,
                            "discard_reason": reason
                        }
                        # Conservar el mejor intento si aún no había uno o si mejoró cobertura
                        if best_candidates[idx] is None or keep or (stats["ent_cov"] > best_candidates[idx]["filter_stats"]["ent_cov"]):
                            best_candidates[idx] = cand
                        if not keep:
                            still_failing.append(idx)

                    active_indices = still_failing

                for cand in best_candidates:
                    fout.write(json.dumps(cand, ensure_ascii=False) + "\n")
                fout.flush()

        # Liberar VRAM antes de cargar el siguiente LLM en la T4
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

Writing src/paperizar/informalize.py


Celda 3: Paso A — Ejecutar el Piloto obligatorio de 10 abstracts por generador y auditar tasas de aprobación

Siguiendo estrictamente la Sección "Cómo trabajar en Colab" (Página 19) y la Sección 4 (Paso P3: si algún generador descarta más del 30%, lo detectamos aquí en pocos minutos antes de lanzar los 900 borradores completos):

In [3]:
import json, importlib
import pandas as pd
import src.paperizar.informalize as inf_mod
importlib.reload(inf_mod)

with open("data/processed/roundtrip.jsonl", encoding="utf-8") as f:
    rt_all = [json.loads(line) for line in f]

# Piloto: 10 abstracts x 3 generadores x 1 nivel representativo (nivel 2) = 30 borradores rápidos (~2.5 min)
pilot_records = rt_all[:10]
inf_mod.run_informalization(
    input_records=pilot_records,
    out_jsonl="data/processed/pilot_informal.jsonl",
    generators_to_run=["G1", "G2", "G3"],
    levels=[2],
    batch_size=5,
    max_attempts=2
)

# Auditar tasas de keep y motivos de descarte por generador
with open("data/processed/pilot_informal.jsonl", encoding="utf-8") as f:
    pilot_out = [json.loads(line) for line in f]

df_p = pd.DataFrame(pilot_out)
summary_p = df_p.groupby("generator").agg(
    total=("keep", "count"),
    aprobados=("keep", "sum"),
    tasa_keep=("keep", "mean"),
    cov_media=("filter_stats", lambda x: round(sum(d["ent_cov"] for d in x)/len(x), 3)),
    rougeL_medio=("filter_stats", lambda x: round(sum(d["rougeL_AI"] for d in x)/len(x), 3)),
).reset_index()

print("\n=== AUDITORÍA DEL PILOTO (10 ABSTRACTS POR GENERADOR) ===")
print(summary_p.to_string(index=False))

print("\n--- Muestra de 1 borrador informal generado (keep=True) ---")
ok_samples = [r for r in pilot_out if r["keep"]]
if ok_samples:
    s = ok_samples[0]
    print(f"[{s['generator']} | Nivel {s['level']} | Stats: {s['filter_stats']}]")
    print("ORIGINAL (A):", s["A"][:280], "...")
    print("INFORMAL (I):", s["I"][:280], "...")
else:
    print("Descartes detectados:", df_p["discard_reason"].value_counts().to_dict())

[data/processed/pilot_informal.jsonl] Registros previos detectados para reanudar: 0

Cargando G1: microsoft/Phi-3.5-mini-instruct (10 borradores pendientes)...


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

... (31 outputs left out)

generation_config.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

G3:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



=== AUDITORÍA DEL PILOTO (10 ABSTRACTS POR GENERADOR) ===
generator  total  aprobados  tasa_keep  cov_media  rougeL_medio
       G1     10         10        1.0      0.957         0.386
       G2     10          2        0.2      0.875         0.643
       G3     10         10        1.0      0.966         0.522

--- Muestra de 1 borrador informal generado (keep=True) ---
[G1 | Nivel 2 | Stats: {'ent_cov': 0.9444, 'len_ratio': 0.9609, 'rougeL_AI': 0.3359}]
ORIGINAL (A): Bit-interleaved coded modulation (BICM) is widely used in multiple-input multiple-output (MIMO) wireless systems. However, separating MIMO data detection from channel decoding and only exchanging per-bit soft information discards correlation information among the bits within each ...
INFORMAL (I): So we're digging into Bit-interleaved coded modulation (BICM) in MIMO wireless systems, right? But there's a snag. We lose some connection info when we only swap per-bit soft data and skip channel decoding. This messes up our

Celda 4: Actualizar G2 a ibm-granite/granite-3.1-2b-instruct en código y MANIFEST.md, y validar su piloto de 10 textos

In [4]:
import json, hashlib, importlib
import pandas as pd

# 1. Actualizar G2 en src/paperizar/informalize.py y cambiar torch_dtype -> dtype limpio
with open("src/paperizar/informalize.py", "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace(
    '"G2": "HuggingFaceTB/SmolLM2-1.7B-Instruct"',
    '"G2": "ibm-granite/granite-3.1-2b-instruct"'
).replace("torch_dtype=torch.float16", "dtype=torch.float16")

with open("src/paperizar/informalize.py", "w", encoding="utf-8") as f:
    f.write(code)

# Actualizar también data/MANIFEST.md
with open("data/MANIFEST.md", "r", encoding="utf-8") as f:
    man = f.read()
man = man.replace(
    "| **Generador `G2` (Train/Val + Roundtrip)** | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | HF SmolLM2 | 1.71B | 2024-07 |",
    "| **Generador `G2` (Train/Val + Roundtrip)** | `ibm-granite/granite-3.1-2b-instruct` | IBM Granite | 2.53B | 2024-12 |"
)
with open("data/MANIFEST.md", "w", encoding="utf-8") as f:
    f.write(man)

import src.paperizar.informalize as inf_mod
importlib.reload(inf_mod)

# 2. Limpiar en pilot_informal.jsonl solo las 10 filas viejas de G2 y correr el nuevo G2
with open("data/processed/pilot_informal.jsonl", encoding="utf-8") as f:
    kept_pilot = [json.loads(line) for line in f if json.loads(line)["generator"] != "G2"]
with open("data/processed/pilot_informal.jsonl", "w", encoding="utf-8") as f:
    for r in kept_pilot:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open("data/processed/roundtrip.jsonl", encoding="utf-8") as f:
    rt_all = [json.loads(line) for line in f]

inf_mod.run_informalization(
    input_records=rt_all[:10],
    out_jsonl="data/processed/pilot_informal.jsonl",
    generators_to_run=["G2"],
    levels=[2],
    batch_size=5,
    max_attempts=3
)

with open("data/processed/pilot_informal.jsonl", encoding="utf-8") as f:
    pilot_out = [json.loads(line) for line in f]

df_p = pd.DataFrame(pilot_out)
summary_p = df_p.groupby("generator").agg(
    total=("keep", "count"),
    aprobados=("keep", "sum"),
    tasa_keep=("keep", "mean"),
    cov_media=("filter_stats", lambda x: round(sum(d["ent_cov"] for d in x)/len(x), 3)),
    rougeL_medio=("filter_stats", lambda x: round(sum(d["rougeL_AI"] for d in x)/len(x), 3)),
).reset_index()

print("\n=== AUDITORÍA ACTUALIZADA DEL PILOTO (G1, G2=Granite, G3) ===")
print(summary_p.to_string(index=False))

[data/processed/pilot_informal.jsonl] Registros previos detectados para reanudar: 20

Cargando G2: ibm-granite/granite-3.1-2b-instruct (10 borradores pendientes)...


config.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

G2:   0%|          | 0/2 [00:00<?, ?it/s]


=== AUDITORÍA ACTUALIZADA DEL PILOTO (G1, G2=Granite, G3) ===
generator  total  aprobados  tasa_keep  cov_media  rougeL_medio
       G1     10         10        1.0      0.957         0.386
       G2     10          9        0.9      0.957         0.533
       G3     10         10        1.0      0.966         0.522


Celda 5: Generar el conjunto completo data/processed/pairs.jsonl (900 borradores = 100 roundtrip $\times$ 3 LLM $\times$ 3 niveles) + registro en MLflow (T2)Esta celda genera los 900 borradores completos con batch_size=12 y guardado incremental línea por línea en Google Drive. Si en algún momento Colab se desconectara, al volver a ejecutarla retoma exactamente donde iba sin perder ni un registro. Al terminar, guarda la tabla de descartes por generador y nivel (results/tabla_descartes_informal.csv) y registra el hash de pairs.jsonl en MLflow.

In [5]:
import json, hashlib
import pandas as pd
import mlflow
import src.paperizar.informalize as inf_mod

with open("data/processed/roundtrip.jsonl", encoding="utf-8") as f:
    rt_all = [json.loads(line) for line in f]

# Generar los 900 borradores (100 abstracts x 3 generadores x 3 niveles)
# batch_size=12 aprovecha al máximo los 16 GB de la T4 en fp16
inf_mod.run_informalization(
    input_records=rt_all,
    out_jsonl="data/processed/pairs.jsonl",
    generators_to_run=["G1", "G2", "G3"],
    levels=[1, 2, 3],
    batch_size=12,
    max_attempts=3
)

# Auditar resultados completos de pairs.jsonl (Tarea 2)
with open("data/processed/pairs.jsonl", encoding="utf-8") as f:
    all_pairs = [json.loads(line) for line in f]

with open("data/processed/pairs.jsonl", "rb") as fb:
    pairs_sha256 = hashlib.sha256(fb.read()).hexdigest()[:16]

df_pairs = pd.DataFrame(all_pairs)
tab_gen = df_pairs.groupby(["generator", "level"]).agg(
    total=("keep", "count"),
    aprobados=("keep", "sum"),
    tasa_keep=("keep", lambda s: round(s.mean(), 4)),
    cov_media=("filter_stats", lambda x: round(sum(d["ent_cov"] for d in x)/len(x), 4)),
    rougeL_medio=("filter_stats", lambda x: round(sum(d["rougeL_AI"] for d in x)/len(x), 4)),
).reset_index()

tab_gen.to_csv("results/tabla_descartes_informal.csv", index=False)
global_keep = df_pairs["keep"].mean()

# Registrar en MLflow (Sección 5-F)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("paperizar_informal")
with mlflow.start_run(run_name="roundtrip_900_informalization"):
    mlflow.log_params({
        "G1": inf_mod.GENERATORS["G1"],
        "G2": inf_mod.GENERATORS["G2"],
        "G3": inf_mod.GENERATORS["G3"],
        "temperature": 0.8,
        "top_p": 0.95,
        "pairs_jsonl_hash": pairs_sha256,
        "seed": 42
    })
    mlflow.log_metrics({
        "total_pairs": len(df_pairs),
        "kept_pairs": int(df_pairs["keep"].sum()),
        "global_keep_rate": round(float(global_keep), 4)
    })

print(f"\n=== TAREA 2 CERRADA (pairs.jsonl hash={pairs_sha256}) ===")
print(f"Total borradores generados: {len(df_pairs)} | Aprobados (keep=True): {df_pairs['keep'].sum()} ({global_keep*100:.2f}%)")
print("\nDesglose por generador y nivel:")
print(tab_gen.to_string(index=False))

[data/processed/pairs.jsonl] Registros previos detectados para reanudar: 0

Cargando G1: microsoft/Phi-3.5-mini-instruct (300 borradores pendientes)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

G1:   0%|          | 0/25 [00:00<?, ?it/s]


Cargando G2: ibm-granite/granite-3.1-2b-instruct (300 borradores pendientes)...


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

G2:   0%|          | 0/25 [00:00<?, ?it/s]


Cargando G3: tiiuae/Falcon3-3B-Instruct (300 borradores pendientes)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

G3:   0%|          | 0/25 [00:00<?, ?it/s]

2026/11/14 00:36:47 INFO mlflow.tracking.fluent: Experiment with name 'paperizar_informal' does not exist. Creating a new experiment.



=== TAREA 2 CERRADA (pairs.jsonl hash=9c5c3b465530702d) ===
Total borradores generados: 900 | Aprobados (keep=True): 593 (65.89%)

Desglose por generador y nivel:
generator  level  total  aprobados  tasa_keep  cov_media  rougeL_medio
       G1      1    100         83       0.83     0.9508        0.4444
       G1      2    100         79       0.79     0.9411        0.3596
       G1      3    100         47       0.47     0.9232        0.3289
       G2      1    100         85       0.85     0.9725        0.5274
       G2      2    100         76       0.76     0.9778        0.5408
       G2      3    100         63       0.63     0.9522        0.5014
       G3      1    100         73       0.73     0.9562        0.5428
       G3      2    100         67       0.67     0.9523        0.5315
       G3      3    100         20       0.20     0.8167        0.3161
